In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

In [3]:
from pyspark.sql.functions import col, log, when
from src.spark_session import create_spark
from src.utils import raw_file_path, processed_path

## 1) Raw -> Processed : 전처리

- 숫자 컬럼 타입 명시적으로 정리

- 이상한 행/결측치 제거

- 파생 컬럼 여러 개 생성

- processed는 csv 대신 parquet으로 저장

In [4]:
spark = create_spark()

26/03/14 16:43:07 WARN Utils: Your hostname, Bigbreads-MacBook-M2-Pro-Main.local resolves to a loopback address: 127.0.0.1; using 192.168.1.224 instead (on interface en0)
26/03/14 16:43:07 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/14 16:43:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [5]:
# 전처리 함수
def preprocess_one_day(target_date: str):
    # 1) raw 읽기
    df_raw = spark.read.csv(
        raw_file_path(target_date),
        header=True,
        inferSchema=True
    )

    # 2) 타입 정리 + 필요한 컬럼만 선택
    df_clean = (
        df_raw
        .select("Date", "Ticker", "Open", "High", "Low", "Close", "Volume")
        .withColumn("Open", col("Open").cast("double"))
        .withColumn("High", col("High").cast("double"))
        .withColumn("Low", col("Low").cast("double"))
        .withColumn("Close", col("Close").cast("double"))
        .withColumn("Volume", col("Volume").cast("long"))
    )

    # 3) 결측치 제거 + 비정상 데이터 제거
    df_clean = (
        df_clean
        .dropna(subset=["Date", "Ticker", "Open", "High", "Low", "Close", "Volume"])
        .filter(col("Open") > 0)
        .filter(col("Close") > 0)
        .filter(col("High") >= col("Low"))
        .filter(col("Volume") >= 0)
    )

    # 4) 파생 변수 생성
    df_processed = (
        df_clean
        .withColumn("daily_return", (col("Close") - col("Open")) / col("Open"))
        .withColumn("price_range", col("High") - col("Low"))
        .withColumn("range_ratio", (col("High") - col("Low")) / col("Open"))
        .withColumn("log_return", log(col("Close") / col("Open")))
        .withColumn(
            "direction",
            when(col("Close") > col("Open"), "UP")
            .when(col("Close") < col("Open"), "DOWN")
            .otherwise("FLAT")
        )
        .select(
            "Date", "Ticker",
            "Open", "High", "Low", "Close", "Volume",
            "daily_return", "price_range", "range_ratio", "log_return", "direction"
        )
    )

    # 5) parquet 저장
    output_path = f"../data/processed/{target_date}"
    df_processed.write.mode("overwrite").parquet(output_path)

    print(f"Saved processed parquet: {output_path}")
    df_processed.show(truncate=False)

In [6]:
# ✅ 실행 : 3/10 raw 전처리
preprocess_one_day("2026-03-10")

Saved processed parquet: ../data/processed/2026-03-10
+----------+------+------------------+------------------+------------------+------------------+---------+---------------------+------------------+--------------------+---------------------+---------+
|Date      |Ticker|Open              |High              |Low               |Close             |Volume   |daily_return         |price_range       |range_ratio         |log_return           |direction|
+----------+------+------------------+------------------+------------------+------------------+---------+---------------------+------------------+--------------------+---------------------+---------+
|2026-03-10|AAPL  |257.6499938964844 |262.4800109863281 |256.95001220703125|260.8299865722656 |30590800 |0.012342296724675532 |5.529998779296875 |0.021463221076258414|0.012266751545973564 |UP       |
|2026-03-10|MSFT  |410.0299987792969 |410.20001220703125|402.92999267578125|405.760009765625  |31706400 |-0.010413845392737334|7.27001953125     |

In [ ]:
# 이전에 수집한 데이터 "2026-03-03"~"2026-03-08" 전처리
# preprocess_one_day("2026-03-04")
# preprocess_one_day("2026-03-05")
# preprocess_one_day("2026-03-06")
# preprocess_one_day("2026-03-09")

## 2) Processed -> Data Mart

In [7]:
# 1. processed "2026-03-04 ~ 2026-03-10" parquet 전체 읽기
df_all = spark.read.parquet("../data/processed/*")

df_all.show(truncate=False)
df_all.printSchema()

+----------+------+------------------+------------------+------------------+------------------+---------+----------------------+------------------+--------------------+----------------------+---------+
|Date      |Ticker|Open              |High              |Low               |Close             |Volume   |daily_return          |price_range       |range_ratio         |log_return            |direction|
+----------+------+------------------+------------------+------------------+------------------+---------+----------------------+------------------+--------------------+----------------------+---------+
|2026-03-10|AAPL  |257.6499938964844 |262.4800109863281 |256.95001220703125|260.8299865722656 |30590800 |0.012342296724675532  |5.529998779296875 |0.021463221076258414|0.012266751545973564  |UP       |
|2026-03-10|MSFT  |410.0299987792969 |410.20001220703125|402.92999267578125|405.760009765625  |31706400 |-0.010413845392737334 |7.27001953125     |0.01773045765649739 |-0.010468448899809815 |D

In [8]:
# 2. SQL View 만들기
df_all.createOrReplaceTempView("stocks")

## 3) datamart 집계 쿼리 - RDD vs DataFrame

### 1️⃣ 종목별 평균 일일 수익률

In [ ]:
rdd = df_all.select("Ticker", "Close").rdd

max_close_rdd = (
    rdd.map(lambda x: (x["Ticker"], x["Close"]))
       .reduceByKey(lambda a, b: a if a > b else b)
)

max_close_rdd.collect()

[('MSFT', 410.67999267578125),
 ('GOOG', 306.92999267578125),
 ('TSLA', 405.94000244140625),
 ('AAPL', 262.5199890136719),
 ('NVDA', 184.77000427246094)]

### 2️⃣ 종목별 평균 거래량 (과제)

In [12]:
rdd = df_all.select("Ticker", "Close").rdd

max_close_rdd = (
    rdd.map(lambda x: (x["Ticker"], x["Close"]))
       .reduceByKey(lambda a, b: a if a > b else b)
)

max_close_rdd.collect()

[('MSFT', 410.67999267578125),
 ('GOOG', 306.92999267578125),
 ('TSLA', 405.94000244140625),
 ('AAPL', 262.5199890136719),
 ('NVDA', 184.77000427246094)]

In [14]:
from pyspark.sql.functions import max as spark_max, col

df_all.groupBy("Ticker") \
    .agg(spark_max("Close").alias("max_close")) \
    .orderBy(col("max_close").desc()) \
    .show()

+------+------------------+
|Ticker|         max_close|
+------+------------------+
|  MSFT|410.67999267578125|
|  TSLA|405.94000244140625|
|  GOOG|306.92999267578125|
|  AAPL| 262.5199890136719|
|  NVDA|184.77000427246094|
+------+------------------+



In [15]:
spark.sql("""
SELECT
    Ticker,
    MAX(Close) AS max_close
FROM stocks
GROUP BY Ticker
ORDER BY max_close DESC
""").show()

+------+------------------+
|Ticker|         max_close|
+------+------------------+
|  MSFT|410.67999267578125|
|  TSLA|405.94000244140625|
|  GOOG|306.92999267578125|
|  AAPL| 262.5199890136719|
|  NVDA|184.77000427246094|
+------+------------------+



In [21]:
from pyspark.sql.functions import max as spark_max, col

max_close_df = (
    df_all.groupBy("Ticker")
        .agg(spark_max("Close").alias("max_close"))
        .orderBy(col("max_close").desc())
)

max_close_df.show()

max_close_df.write \
    .mode("overwrite") \
    .parquet("../data/datamart/max_close_by_ticker")

+------+------------------+
|Ticker|         max_close|
+------+------------------+
|  MSFT|410.67999267578125|
|  TSLA|405.94000244140625|
|  GOOG|306.92999267578125|
|  AAPL| 262.5199890136719|
|  NVDA|184.77000427246094|
+------+------------------+



In [22]:
spark.read.parquet("../data/datamart/max_close_by_ticker").show()

+------+------------------+
|Ticker|         max_close|
+------+------------------+
|  MSFT|410.67999267578125|
|  TSLA|405.94000244140625|
|  GOOG|306.92999267578125|
|  AAPL| 262.5199890136719|
|  NVDA|184.77000427246094|
+------+------------------+

